## IPS Pain Point Analysis Notebook
This notebook loads sentiment-tagged challenge data, maps pain points to categories, and reports coverage and gaps in a presentation-friendly format.

In [1]:
%pip install -r ../requirements.txt

ERROR: Could not find a version that satisfies the requirement csv (from versions: none)
ERROR: No matching distribution found for csv
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

pd.set_option('display.max_colwidth', 200)

Matplotlib is building the font cache; this may take a moment.


In [3]:
# 1) Load and preprocess datasets
CHALLENGES_PATH = '../output/clean_challenges.csv'
EXPECTATIONS_PATH = '../output/clean_expectations.csv'

def load_and_clean(path, text_col):
    df = pd.read_csv(path, engine='python', on_bad_lines='skip').copy()
    df.columns = df.columns.str.strip()

    # normalize common column variants
    rename_map = {
        'Pain Points': 'Pain Point',
        'Expectations': 'Expectation',
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    if text_col not in df.columns:
        raise KeyError(f"Column '{text_col}' not found. Available: {list(df.columns)}")

    df = df[~df[text_col].str.contains(r'Other|IPS related', case=False, na=False, regex=True)].copy()
    df['word_count'] = df[text_col].str.split().str.len()
    df = df[df['word_count'] > 4].drop(columns=['word_count'])
    return df

df_painpoints = load_and_clean(CHALLENGES_PATH, 'pain_points')
df_expectations = load_and_clean(EXPECTATIONS_PATH, 'expectations')

In [4]:
df_painpoints.sample(5)

,focus_group,pain_points,sentiment
202,DOCE Supervisors,"Paper-Based Archives: Information remains stored in microfiche, paper files, and legacy systems.",neutral
116,CPC CPC,"Mixed Violation Records: BAA tickets and Code Enforcement violations appear together, making it difficult to distinguish between them.",negative
209,DOCE Supervisors,Constituent Escalations: Supervisors spend significant time responding to escalated complaints and requests.,neutral
251,DOCE Fire Prevention Bureau,Multiple Search Locations: Information exists in several areas of the system.,neutral
195,DOCE Supervisors,Volume Challenges: IPS becomes cumbersome as case volume increases.,negative


In [5]:
df_expectations.sample(5)

,focus_group,expectations,sentiment
126,CPC CPC,"Improved Public Experience: Enable staff to answer public questions quickly using complete, accurate information from one system.",positive
191,DOCE Supervisors,"AI-Assisted Research: Use AI to locate related properties, permits, violations, and ownership information.",neutral
78,DOCE Building Inspectors,One System for Everything: Reduce dependency on multiple systems.,positive
125,CPC CPC,"Improved Interdepartmental Coordination: Improve communication and information sharing between CPC, BAA, Finance, Law, and Code Enforcement.",positive
194,DOCE Supervisors,Cloud-Based Access: Allow users to access information from anywhere.,neutral


In [6]:
# # 2) Reusable matching helpers
# def _normalize_keywords(keywords):
#     if keywords is None:
#         return []
#     terms = keywords if isinstance(keywords, list) else [keywords]
#     return [str(term).strip() for term in terms if str(term).strip()]


# def _build_keyword_mask(series, keywords):
#     terms = _normalize_keywords(keywords)
#     if not terms:
#         return pd.Series(True, index=series.index)
#     pattern = '|'.join(map(re.escape, terms))
#     return series.str.contains(pattern, case=False, na=False, regex=True)


# def _normalize_sentiments(sentiment):
#     if sentiment is None:
#         return []
#     sentiments = sentiment if isinstance(sentiment, list) else [sentiment]
#     return [str(item).strip() for item in sentiments if str(item).strip()]

# def _build_search_text(df, primary_col='processed_content', fallback_col='Pain Point'):
#     primary = df[primary_col].fillna('').astype(str) if primary_col in df.columns else pd.Series('', index=df.index)
#     fallback = df[fallback_col].fillna('').astype(str) if fallback_col in df.columns else pd.Series('', index=df.index)
#     return (primary + ' ' + fallback).str.strip()


# def filter_pain_points(df, keyword, optional_keywords=None, sentiment=None):
#     search_text = _build_search_text(df, primary_col='processed_content', fallback_col='Pain Point')
#     required_mask = _build_keyword_mask(search_text, keyword)
#     filtered = df[required_mask].copy()

#     sentiment_terms = _normalize_sentiments(sentiment)
#     if sentiment_terms:
#         filtered = filtered[filtered['label'].isin(sentiment_terms)].copy()

#     optional_terms = _normalize_keywords(optional_keywords)
#     if optional_terms and not filtered.empty:
#         optional_pattern = '|'.join(map(re.escape, optional_terms))
#         rank_text = _build_search_text(filtered, primary_col='processed_content', fallback_col='Pain Point')
#         filtered['_optional_match_count'] = rank_text.str.count(optional_pattern, flags=re.IGNORECASE)
#         filtered = filtered.sort_values(
#             by=['_optional_match_count', 'Focus Group', 'Pain Point'],
#             ascending=[False, True, True]
#         ).drop(columns=['_optional_match_count'])

#     return filtered

In [ ]:
categories = {
    "IPS crash" : ["crash", "crashes", "crashed", "freezes", "freeze", 
                    "frozen", "hangs", "hang", "stuck", "stuck in", "slow", "lag", "lags", "laggy", 
                    "unresponsive", "unresponsiveness", "slow response", "slow responses"],
    "Fragmented Data" : ["fragmented", "fragmentation", "disjointed", "disconnected", "inconsistent",
                        "incoherent", "scattered", "dispersed", "isolated",
                        "data silos", "data silo", "data fragmentation", "data inconsistency"],
    "System Integration" : ["integration", "integrated", "interoperability", "interoperable", 
                        "system integration", "system integrations", "platform integration", 
                        "platform integrations", "software integration", "software integrations"],
    "User Interface" : ["UI", "user interface", "UX", "user experience", "navigation", 
                        "layout", "design", "usability", "user-friendly",
                        "intuitive", "responsive design", "aesthetic", "visual design", 
                        "interaction design", "user-centered design", "accessibility", "mobile-friendly", "cross-platform"],
    "Manual Data Entry" : ["manual entry", "manual input", "data entry", "data input", 
                        "manual data entry", "manual data input", "human error", 
                        "error-prone", "time-consuming", "tedious", "repetitive tasks"],
    "In field Usability" : ["in-field usability", "field usability", "on-site usability",
                        "field operations", "on-site operations", "field work",
                        "on-site work", "field tasks", "on-site tasks", "field challenges",
                        "on-site challenges", "field environment", "on-site environment",
                        "field conditions", "on-site conditions", "field performance",
    "Data Visibility" : ["data visibility", "data transparency", "data access", "data sharing",
                        "data availability", "data insights", "data reporting", 
                        "data analytics", "data-driven decisions", "real-time data",
                        "real-time insights", "real-time reporting", "real-time analytics"],
    "Reporting and Analytics" : ["reporting", "analytics", "data analysis", "data visualization",
                        "business intelligence", "data-driven insights", "data-driven decisions",
                        "performance metrics", "key performance indicators", "KPIs",
                        "data dashboards", "data reporting tools", "data analytics tools"],
    ""
}

In [ ]:
# categories = {
#     "Data Management": [
#         "data", "information", "record", "records",
#         "database", "history", "historical", "document", "documents",
#         "ownership", "owner", "property", "parcel", "address", "contact",
#         "lookup", "search", "retrieve", "retrieval", "duplicate",
#         "duplicates", "missing", "archive", "archived",
#         "centralized", "centralised", "source of truth", "attachment", "file", "folder",
#         "notes", "case history", "property history", "document storage", "duplicate entry",
#         "re-enter", "reenter", "attachments", "upload", "download", "export", "import"
#     ],

#     "System Integration": [
#         "integration", "integrate", "sync", "synchronize",
#         "camino", "ips", "gis", "esri", "api",
#         "interface", "import", "export", "shared",
#         "cross system", "third party", "permit",
#         "assessment", "finance", "mapping", "automation", "linked",
#         "connection", "connected", "multiple systems", "multiple"
#     ],

#     "Manual Processes": [
#         "manual", "manually", "paper", "spreadsheet",
#         "excel", "email", "phone", "call",
#         "tracking", "track", "entry",
#         "copy", "paste", "duplicate work",
#         "retype", "re-enter", "reenter",
#         "handwritten", "print", "scan",
#         "notification", "calendar",
#         "sticky note", "paperwork", "physical file", "alerts"
#     ],

#     "Workflow Efficiency": [
#         "workflow", "workflows", "process", "approval", "approvals",
#         "routing", "queue", "backlog", "delay",
#         "waiting", "handoff", "review",
#         "reviewer", "assignment", "assigned",
#         "task", "step", "follow up", "follow-up",
#         "escalation", "bottleneck", "inefficiency",
#         "inefficient", "slow process",
#         "steps", "duplicate effort", "rework",
#         "scheduling", "inspection", "inspection scheduling",
#         "case assignment", "workload", "module"
#     ],

#     "Usability": [
#         "navigation", "navigate", "screen",
#         "menu", "button", "click",
#         "too many clicks", "confusing",
#         "difficult", "hard to use",
#         "can't find", "cannot find",
#         "hard to locate", "search",
#         "filter", "sorting",
#         "user friendly", "user-friendly",
#         "layout", "interface",
#         "lookup", "find", "locate",
#         "dropdown", "form", "field", "internet", "connect", 
#         "hotspot", "laptop", "mobile", "tablet", "device", 
#         "browser", "chrome", "edge", "safari", "firefox", "Tickle Dates"
#     ],

#     "System Performance": [
#         "slow", "slowness", "lag",
#         "performance", "crash", "crashes",
#         "freeze", "frozen", "timeout",
#         "loading", "load", "unresponsive",
#         "latency", "error", "bug",
#         "glitch", "fails", "failure",
#         "attachment", "upload", "download",
#         "response time", "speed", "shutdown", "shut down", "shutdowns"
#     ],

#     "Communication": [
#         "communication", "notify", "notification",
#         "email", "call", "phone",
#         "status request", "coordination",
#         "collaboration", "stakeholder",
#         "department", "cross department",
#         "cross-department", "shared information",
#         "meeting", "follow up",
#         "respond", "response",
#         "feedback", "clarification"
#     ],

#     "Visibility & Transparency": [
#         "visibility", "transparent", "transparency",
#         "status", "tracking", "track",
#         "progress", "dashboard",
#         "monitor", "shared",
#         "ownership", "responsible",
#         "review status", "case status",
#         "real time", "real-time",
#         "where is", "who has",
#         "pending", "current status"
#     ],

#     "Data Quality": [
#         "incorrect", "wrong", "error",
#         "duplicate", "duplicates",
#         "missing", "accuracy", "accurate",
#         "quality", "outdated",
#         "invalid", "incomplete",
#         "conflicting", "mismatch",
#         "consistent", "inconsistent",
#         "stale", "obsolete",
#         "clean", "cleanup"
#     ],

#     "Reporting & Analytics": [
#         "report", "reports", "reporting",
#         "dashboard", "analytics",
#         "metrics", "statistics",
#         "trend", "summary",
#         "export", "kpi", "performance measure",
#         "business intelligence", "bi",
#         "filter", "pivot",
#         "ad hoc", "analysis"
#     ],

#     "Compliance & Documentation": [
#         "compliance", "policy",
#         "documentation", "audit",
#         "inspection", "legal",
#         "regulation", "requirement",
#         "required", "evidence",
#         "retention", "code",
#         "ordinance", "standard",
#         "procedure", "guideline",
#         "compliant", "violation"
#     ],

#     "Customer Experience": [
#         "Customers", "customer", "citizen",
#         "resident", "contractor",
#         "applicant", "owner",
#         "public", "service",
#         "complaint", "request",
#         "application", "permit application",
#         "status request", "response time",
#         "waiting", "frustration",
#         "renewal", "inspection request",
#         "phone call", "walk in",
#         "walk-in", "portal"
#     ],
#     "Resource Constraints": [
#         "staff", "staffing", "capacity", "workload",
#         "short staffed", "short-staffed", "resource",
#         "resources", "time", "hours",
#         "bandwidth", "overloaded", "understaffed",
#         "vacancy", "training", "knowledge",
#         "experience", "expertise"
#     ],
#     "Business Rules & Policy": [
#         "business rule", "policy", "ordinance",
#         "requirement", "approval rule",
#         "exception", "eligibility", "criteria",
#         "priority", "classification", "zoning",
#         "code enforcement", "inspection rule",
#         "permit rule", "workflow rule"
#     ]
# }

In [8]:
# Weighted categorization: choose one primary category per record so counts do not overlap.
IMPORTANT_PHRASE_WEIGHTS = {
    'source of truth': 8,
    'single source of truth': 9,
    'multiple systems': 7,
    'cross department': 6,
    'cross-department': 6,
    'status request': 5,
    'response time': 5,
    'duplicate entry': 5,
    'document storage': 5,
    'case history': 5,
    'property history': 5,
    'real-time': 5,
    'real time': 5,
    'business intelligence': 5
}

def _term_weight(term):
    term = str(term).lower().strip()
    if term in IMPORTANT_PHRASE_WEIGHTS:
        return IMPORTANT_PHRASE_WEIGHTS[term]
    # Multi-word phrases carry more context than single tokens.
    return 3 if ' ' in term else 1


def categorize(text, min_score=1):
    text = str(text).lower()

    category_scores = {}
    for category, keywords in categories.items():
        score = 0
        for term in keywords:
            token = str(term).lower().strip()
            if not token:
                continue

            if ' ' in token:
                matched = token in text
            else:
                matched = bool(re.search(rf'\b{re.escape(token)}\b', text))

            if matched:
                score += _term_weight(token)

        if score >= min_score:
            category_scores[category] = score

    if not category_scores:
        return ['Other']

    # Pick the highest-weight category once so each record is counted in only one bucket.
    best_category = max(category_scores.items(), key=lambda item: item[1])[0]
    return [best_category]

In [9]:
def build_clickable_heatmap(source_df, focus_col, text_col, title):
    source_rows = source_df.copy()
    if "Categories" not in source_rows.columns:
        source_rows["Categories"] = source_rows[text_col].apply(categorize)

    heatmap_df = source_rows[[focus_col, "Categories", text_col]].copy()
    heatmap_df = heatmap_df.explode("Categories", ignore_index=True)
    heatmap_df[focus_col] = heatmap_df[focus_col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan})
    heatmap_df["Categories"] = heatmap_df["Categories"].astype(str).str.strip().replace({"": np.nan, "nan": np.nan})
    heatmap_df = heatmap_df.dropna(subset=[focus_col, "Categories"]).reset_index(drop=True)

    all_focus_groups = (
        source_rows[focus_col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan})
        .dropna()
        .unique()
        .tolist()
    )
    all_categories = (
        source_rows["Categories"]
        .explode()
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan})
        .dropna()
        .unique()
        .tolist()
    )

    full_grid = pd.MultiIndex.from_product(
        [all_focus_groups, all_categories],
        names=[focus_col, "Categories"]
    )
    count_series = (
        heatmap_df.groupby([focus_col, "Categories"]).size().reindex(full_grid, fill_value=0)
    )
    count_df = count_series.reset_index(name="record_count")
    pivot = (
        count_df.pivot(index=focus_col, columns="Categories", values="record_count")
        .fillna(0)
        .astype(int)
    )

    row_order = pivot.sum(axis=1).sort_values(ascending=False).index
    col_order = pivot.sum(axis=0).sort_values(ascending=False).index
    pivot = pivot.loc[row_order, col_order]

    fig = go.FigureWidget(
        data=[
            go.Heatmap(
                z=pivot.to_numpy(),
                x=pivot.columns.tolist(),
                y=pivot.index.tolist(),
                colorscale="YlOrRd",
                text=pivot.to_numpy(),
                texttemplate="%{text}",
                hovertemplate="<b>%{y}</b><br>Category: %{x}<br>Records: %{z}<extra></extra>",
                showscale=True,
            )
        ]
    )
    fig.update_layout(
        title=title,
        xaxis_title="Category",
        yaxis_title="Focus Group",
        height=650,
        width=1000,
        margin=dict(l=80, r=30, t=70, b=80),
    )

    source_rows["_focus_clean"] = (
        source_rows[focus_col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan})
    )
    source_rows["_category_list"] = source_rows["Categories"].apply(
        lambda values: [str(item).strip() for item in values if str(item).strip()]
        if isinstance(values, (list, tuple, set))
        else []
    )
    preview_columns = [
        column for column in [focus_col, text_col, "label", "Categories"]
        if column in source_rows.columns
    ]

    output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="0.5rem"))

    def _matching_rows(clicked_focus, clicked_category):
        focus_value = str(clicked_focus).strip()
        category_value = str(clicked_category).strip()
        matched = source_rows[
            source_rows["_focus_clean"].eq(focus_value)
            & source_rows["_category_list"].apply(lambda values: category_value in values)
        ].copy()
        return matched.drop(columns=["_focus_clean", "_category_list"], errors="ignore")

    def _handle_click(trace, points, selector):
        with output:
            output.clear_output()
            clicked_focus = None
            clicked_category = None

            if hasattr(points, "xs") and hasattr(points, "ys") and len(points.xs) > 0 and len(points.ys) > 0:
                clicked_category = points.xs[0]
                clicked_focus = points.ys[0]
            elif getattr(points, "point_inds", None):
                point_index = points.point_inds[0]
                row_index, col_index = divmod(point_index, len(pivot.columns))
                clicked_focus = pivot.index[row_index]
                clicked_category = pivot.columns[col_index]

            if clicked_focus is None or clicked_category is None:
                print("Click a heatmap cell to inspect the matching records.")
                return

            matched_rows = _matching_rows(clicked_focus, clicked_category)
            print(f"Records matched: {len(matched_rows)}")
            print(f"Focus Group: {clicked_focus} | Category: {clicked_category}")

            if matched_rows.empty:
                print("No source rows matched this cell.")
            elif preview_columns:
                display(matched_rows[preview_columns])
            else:
                display(matched_rows)

    fig.data[0].on_click(_handle_click)
    display(widgets.VBox([fig, output]))
    return fig

In [10]:
df_painpoints["Categories"] = df_painpoints["pain_points"].apply(categorize)

print(df_painpoints.head())

                 focus_group  \
0  CPO Central Permit Office   
1  CPO Central Permit Office   
2  CPO Central Permit Office   
3  CPO Central Permit Office   
4  CPO Central Permit Office   

                                                                                                                         pain_points  \
0                     Split Permit & Code Systems: Permit and Code Enforcement information exists in separate systems and workflows.   
1  Scattered Information: Property and permit information is distributed across IPS, Camino, AS400, OnBase, and historical archives.   
2                                   Research Complexity: Users must search multiple systems to understand complete property history.   
3                 Difficult Information Retrieval: Finding relevant information often feels like "hunting" through multiple systems.   
4                                              Limited IPS-Camino Integration: Information does not flow seamlessly between sys

In [11]:
# display records with no categories
display(df_painpoints[df_painpoints['Categories'].apply(lambda x: 'Other' in x)])
#count number of records in each category
category_counts = df_painpoints.explode('Categories')['Categories'].value_counts()
print(category_counts)

,focus_group,pain_points,sentiment,Categories
162,DOCE Admin Aide,Some features behave differently across modules.,neutral,[Other]
177,DOCE Admin Aide,Calls automatically disconnect after several minutes.,neutral,[Other]
182,DOCE Admin Aide,Front-counter traffic can become overwhelming.,neutral,[Other]
243,DOCE Office Manager,Tickets are frequently returned for minor corrections.,neutral,[Other]
248,DOCE Office Manager,Executive support and special projects compete with operational responsibilities.,neutral,[Other]


Categories
Data Management               126
System Integration             59
Manual Processes               25
Workflow Efficiency            23
Usability                      23
Communication                   9
Visibility & Transparency       8
Resource Constraints            8
System Performance              8
Customer Experience             5
Other                           5
Business Rules & Policy         3
Reporting & Analytics           3
Compliance & Documentation      3
Data Quality                    1
Name: count, dtype: int64


In [16]:
# Heatmap: focus group vs pain point category
focus_candidates = ["Focus Group", "focus_group", "focus group"]
focus_col = next((col for col in focus_candidates if col in df_painpoints.columns), None)

if focus_col is None:
    raise KeyError(f"Focus group column not found. Available columns: {list(df_painpoints.columns)}")

fig3 = build_clickable_heatmap(
    df_painpoints,
    focus_col,
    "pain_points",
    "Focus Group vs Pain Point Category"
    )

    'data': [{'colorscale': [[0.0, 'rgb(255,255,204)'], [0.125,
                …

In [13]:
# Heatmap: focus group vs pain point category
focus_candidates = ["Focus Group", "focus_group", "focus group"]
focus_col = next((col for col in focus_candidates if col in df_painpoints.columns), None)

if focus_col is None:
    raise KeyError(f"Focus group column not found. Available columns: {list(df_painpoints.columns)}")

fig3 = build_clickable_heatmap(
    df_painpoints,
    focus_col,
    "pain_points",
    "Focus Group vs Pain Point Category"
    )

    'data': [{'colorscale': [[0.0, 'rgb(255,255,204)'], [0.125,
                …

In [14]:
# Heatmap: focus group vs expectations category
focus_candidates = ["Focus Group", "focus_group", "focus group"]
focus_col = next((col for col in focus_candidates if col in df_expectations.columns), None)

if focus_col is None:
    raise KeyError(f"Focus group column not found. Available columns: {list(df_expectations.columns)}")

fig3 = build_clickable_heatmap(
    df_expectations,
    focus_col,
    "expectations",
    "Focus Group vs Expectations Category"
    )

    'data': [{'colorscale': [[0.0, 'rgb(255,255,204)'], [0.125,
                …